Summary: on benchmark le dataloader

In [1]:
from retinotopy import *
welcome()

---------------------------------------------------------------------------------
On date 2025-11-13, Running learning on host gaia with device mps, pytorch==2.9.1
---------------------------------------------------------------------------------
Welcome on macOS-26.1-arm64-arm-64bit-Mach-O


# Loading legacy images

In [2]:
args = Params()
data_set_type = 'full'
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
args.folders = ['train', 'val'] # type of images to use
args

Params(datetag='2025-11-13', loader='data/Imagenet_urls_ILSVRC_2016.json', annotations_animal='data/Animal10k_annotations.json', annotations_train='data/LOC_train_solution.csv', annotations_val='data/LOC_val_solution.csv', folders=['train', 'val'], tasks=['animal', 'dog', 'cat', 'bird'], image_size=224, num_epochs=20, n_train_stop=0, seed=1998, batch_size=250, batch_size_val=250, lr_conv=1e-05, lr_class=0.001, mutnemom=0.1, ateb2=0.001, weight_decay=0.01, label_smoothing=0.01, rs_min=0.0, rs_max=-5.0, do_polar=True, do_raw=False, do_translate=False, do_resize=True, do_mask=True, do_scratch=False, do_rotation=False, resolution=(11, 11), size_ratio=0.1, do_saccade=False, do_zoom=False, method='valid', saccade_type='multi', normalize=True, verbose=False)

In [3]:
tic = time.time()
args.folders = ['train', 'val'] # type of images to use
dataloaders = datasets_transforms(args)
len(dataloaders['train']), len(dataloaders['train'].dataset)
toc = time.time()
print(f'Loading time for both dataloaders takes \t {toc-tic:.1f} s')  

Loaded 1281166 images under train
Loaded 50000 images under val
Loading time for both dataloaders takes 	 1.4 s


In [4]:
tic = time.time()
args.folders = ['val'] # type of images to use
dataloaders = datasets_transforms(args)
len(dataloaders['val']), len(dataloaders['val'].dataset)
toc = time.time()
print(f'Loading time theval  dataloader takes \t {toc-tic:.1f} s')  

Loaded 50000 images under val
Loading time theval  dataloader takes 	 0.1 s


Benchmarking different methods for the dataloader:

In [ ]:
for num_workers_ in [0, 1, 2, 4, 8, 16 , 32]: # , 16 , 32
    for batch_size_ in [1, 4, 16, 32, 128, 256, 512, 1024, ]: # 2048
        for pin_memory_ in [True, False]: # [False]: #
            args = Params()
            args.batch_size = batch_size_
            args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
            args.folders = ['val'] # type of images to use            
        
            dataloaders = datasets_transforms(args, pin_memory=pin_memory_, num_workers=num_workers_, verbose=False)
            tic = time.time()
            i_image, i_image_max = 0, 2**13
            for i_step, (images, labels) in enumerate(dataloaders['val']):
                    images, labels = images.to(device), labels.to(device)
                    i_image += len(images)
                    if i_image > i_image_max:
                        break

            toc = time.time()
            print(f'{pin_memory_=} \t {num_workers_=} \t {batch_size_=:04d} \t Loading time for {i_image_max} images \t {toc-tic:.1f} s')  

pin_memory_=False 	 num_workers_=0 	 batch_size_=0001 	 Loading time for 8192 images 	 20.5 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0004 	 Loading time for 8192 images 	 20.5 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0016 	 Loading time for 8192 images 	 20.2 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0032 	 Loading time for 8192 images 	 20.1 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0128 	 Loading time for 8192 images 	 20.3 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0256 	 Loading time for 8192 images 	 20.8 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=0512 	 Loading time for 8192 images 	 19.9 s
pin_memory_=False 	 num_workers_=0 	 batch_size_=1024 	 Loading time for 8192 images 	 20.3 s
pin_memory_=False 	 num_workers_=1 	 batch_size_=0001 	 Loading time for 8192 images 	 22.0 s
pin_memory_=False 	 num_workers_=1 	 batch_size_=0004 	 Loading time for 8192 images 	 21.0 s
pin_memory_=False 	 num_workers_=1 	 batch_size_=0016 	 Load

KeyboardInterrupt: 

libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/laurent/sdrive_cnrs/hot_from_git/Retinotopy_project/Retinotopy/.venv/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
    ~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/laurent/sdrive_cnrs/hot_from_git/Retinotopy_project/Retinotopy/.venv/lib/python3.13/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
    ~~~~~~~~~^^
  File "/Users/laurent/sdrive_cnrs/hot_from_git/Retinotopy_project/Retinotopy/.venv/lib/python3.13/site-packages/ipykernel/kernelapp.py", line 758, in sta

: 

In [ ]:
model_filename = f'cached_data/{datetag}_full_resnet101_retino.pt'
model = load_model(model_name='resnet101', model_path=model_filename, do_scratch=False, do_circular=False, verbose=True).to(device)

N_test = 2**8
for num_workers_ in [0, 1, 2, 5, 8, 16 , 32]: # , 16 , 32
    for batch_size_ in [1, 4, 16, 32, 64, 128, 256, 512]: #, 1024, 2048]:
        for pin_memory_ in [False]: #[True, False]: # 
            args = Params()
            args.batch_size_val = batch_size_
            args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
            args.folders = ['val'] # type of images to use            
        
            dataloaders = datasets_transforms(args, pin_memory=pin_memory_, num_workers=num_workers_, verbose=False)
            tic = time.time()
            for i_step, (images, labels) in enumerate(dataloaders['val']):
                    images, labels = images.to(device), labels.to(device)
                    with torch.no_grad():
                        outputs = model(images)
                    if i_step > N_test/batch_size_: break
            toc = time.time()
            print(f'{pin_memory_=} \t\t {num_workers_=} \t\t {batch_size_=:03d} \t\t Elapsed time per image: {1000*(toc-tic)/N_test:.1f} ms')  